In [1]:
# train_vqvae_cifar10.py
# 使用 CVQVAE 在 CIFAR-10 上进行训练和可视化重建

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils
from tqdm import tqdm
import os

from models.cvqvae import CVQVAE


In [2]:
# ----- 设置参数 -----
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 20
LR = 2e-4
IMAGE_SIZE = 64
SAVE_DIR = "samples_cifar10"
os.makedirs(SAVE_DIR, exist_ok=True)

# ----- CIFAR-10 数据加载 -----
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ----- 模型与优化器 -----
model = CVQVAE(num_embeddings=512, embedding_dim=64).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

Files already downloaded and verified


In [3]:



# ----- 训练主循环 -----
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x, _ in tqdm(dataloader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        x = x.to(device)
        optimizer.zero_grad()
        recon, vq_loss = model(x, x)
        recon_loss = F.mse_loss(recon, x)
        loss = recon_loss + vq_loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1} Loss: {avg_loss:.4f}")

    # 保存样例图像
    model.eval()
    with torch.no_grad():
        recon, _ = model(x[:8], x[:8])
        grid = utils.make_grid(torch.cat([x[:8].cpu(), recon.cpu()], dim=0), nrow=8)
        utils.save_image(grid, f"{SAVE_DIR}/epoch{epoch+1}.png")


Epoch 1/20:   0%|          | 0/1563 [00:00<?, ?it/s]

Epoch 1/20: 100%|██████████| 1563/1563 [00:45<00:00, 34.71it/s]


Epoch 1 Loss: 0.1346


Epoch 2/20: 100%|██████████| 1563/1563 [00:44<00:00, 35.16it/s]


Epoch 2 Loss: 0.0175


Epoch 3/20: 100%|██████████| 1563/1563 [00:44<00:00, 34.83it/s]


Epoch 3 Loss: 0.0101


Epoch 4/20:   4%|▎         | 55/1563 [00:01<00:48, 31.01it/s]


KeyboardInterrupt: 